In [1]:
import os
import fiona
import geopandas as gpd
import pandas as pd
import numpy as np
import pyarrow as pa

In [2]:
gml_dir = "./maps/ALKIS_Liegenschaftskarte_ausgewaehlteDaten_HH_2026-01-15/" 

# All available layers in ALKIS GML files (from a single example so there is more in other files probs.)
ALKIS_LAYERS = [
    'AP_LPO', 'AP_PPO', 'AP_PTO',
    'AX_Baublock', 'AX_BauRaumOderBodenordnungsrecht', 'AX_Bauteil',
    'AX_BauwerkImGewaesserbereich', 'AX_BauwerkImVerkehrsbereich',
    'AX_BauwerkOderAnlageFuerSportFreizeitUndErholung',
    'AX_BesondereFlurstuecksgrenze', 'AX_BesondereGebaeudelinie',
    'AX_Bodenschaetzung', 'AX_DammWallDeich', 'AX_Denkmalschutzrecht',
    'AX_FlaecheBesondererFunktionalerPraegung', 'AX_FlaecheGemischterNutzung',
    'AX_Fliessgewaesser', 'AX_Flurstueck', 'AX_Gebaeude', 'AP_Darstellung',
    'AX_Gehoelz', 'AX_GeoreferenzierteGebaeudeadresse',
    'AX_GrablochDerBodenschaetzung', 'AX_IndustrieUndGewerbeflaeche',
    'AX_KlassifizierungNachWasserrecht', 'AX_Landwirtschaft', 'AX_Platz',
    'AX_Schiffsverkehr', 'AX_SonstigesBauwerkOderSonstigeEinrichtung',
    'AX_SonstigesRecht', 'AX_SportFreizeitUndErholungsflaeche',
    'AX_StehendesGewaesser', 'AX_Strassenverkehr', 'AX_Strassenverkehrsanlage',
    'AX_Strukturlinie3D', 'AX_UnlandVegetationsloseFlaeche',
    'AX_UntergeordnetesGewaesser', 'AX_Vegetationsmerkmal',
    'AX_VorratsbehaelterSpeicherbauwerk', 'AX_Wald', 'AX_Weg',
    'AX_WegPfadSteig', 'AX_Wohnbauflaeche'
]

# Columns available in AX_Gebaeude (from a single example so they are diff. in other files probs.)
AX_GEBAEUDE_COLUMNS = [
    'identifier', 'beginnt', 'advStandardModell', 'gebaeudefunktion',
    'weitereGebaeudefunktion', 'name', 'bauweise', 'anzahlDerOberirdischenGeschosse',
    'anzahlDerUnterirdischenGeschosse', 'hochhaus', 'objekthoehe',
    'dachform', 'zustand', 'baujahr', 'lagezurErdoberflaeche',
    'dachart', 'dachgeschossausbau', 'description', 'geometry'
]

In [3]:
def extract_alkis_gdf(gml_dir: str, layer: str = "AX_Gebaeude",
    columns: list[str] | None = None,        # None = keep all; list = keep only these + geometry
    search_token: bytes | None = None,        # None = auto-derive from layer name
    crs: str = "EPSG:25832", print_hits: bool = False, print_progress: bool = True, ) -> gpd.GeoDataFrame:
    """
    Scan all GML/XML files in gml_dir, find those containing `layer`,
    read that layer from each, and return a single concatenated GeoDataFrame.

    Parameters
    ----------
    gml_dir       : directory with ALKIS .xml / .gml files
    layer         : ALKIS layer name to extract (see ALKIS_LAYERS)
    columns       : columns to keep in the output (None = all); geometry always included
    search_token  : bytes to grep for when scanning files; defaults to layer name encoded
    crs           : CRS to assign (data has none stored); ALKIS Hamburg = EPSG:25832
    print_hits    : print filenames that contain the layer
    print_progress: print a progress line per file
    """
    token = search_token or layer.encode()

    # --- 1. find files containing the layer ---
    hits = []
    for fname in sorted(os.listdir(gml_dir)):
        if not fname.endswith((".xml", ".gml")):
            continue
        fpath = os.path.join(gml_dir, fname)
        found = False
        with open(fpath, "rb") as f:
            while not found:
                chunk = f.read(512_000)
                if not chunk:
                    break
                if token in chunk:
                    found = True
        if found:
            hits.append(fpath)

    if print_hits:
        print(f"Files containing '{layer}': {len(hits)}")
        for h in hits:
            print(f"  {os.path.basename(h)}")

    # --- 2. read layer from each file and collect ---
    frames = []
    for i, fpath in enumerate(hits, 1):
        if print_progress:
            print(f"[{i}/{len(hits)}] {os.path.basename(fpath)}", end=" ... ")
        try:
            gdf = gpd.read_file(fpath, layer=layer)

            # column selection (always keep geometry)
            if columns is not None:
                keep = [c for c in columns if c in gdf.columns]
                if "geometry" not in keep:
                    keep.append("geometry")
                gdf = gdf[keep]

            frames.append(gdf)
            if print_progress:
                print(f"{len(gdf)} rows")
        except Exception as e:
            if print_progress:
                print(f"SKIPPED ({e})")

    if not frames:
        raise ValueError(f"No data found for layer '{layer}' in {gml_dir}")

    # --- 3. concatenate and assign CRS ---
    result = pd.concat(frames, ignore_index=True)
    result = gpd.GeoDataFrame(result, geometry="geometry")
    result = result.set_crs(crs)

    print(f"\nDone. Total rows: {len(result):,} | Columns: {list(result.columns)}")
    return result

In [4]:
gdf = extract_alkis_gdf(gml_dir)
print(gdf.shape)

[1/229] HmbTG_ALKIS_260115_031von283.xml ... 769 rows
[2/229] HmbTG_ALKIS_260115_032von283.xml ... 251 rows
[3/229] HmbTG_ALKIS_260115_033von283.xml ... 366 rows
[4/229] HmbTG_ALKIS_260115_035von283.xml ... 23 rows
[5/229] HmbTG_ALKIS_260115_036von283.xml ... 60 rows
[6/229] HmbTG_ALKIS_260115_037von283.xml ... 572 rows
[7/229] HmbTG_ALKIS_260115_039von283.xml ... 707 rows
[8/229] HmbTG_ALKIS_260115_040von283.xml ... 854 rows
[9/229] HmbTG_ALKIS_260115_041von283.xml ... 1061 rows
[10/229] HmbTG_ALKIS_260115_042von283.xml ... 608 rows
[11/229] HmbTG_ALKIS_260115_043von283.xml ... 30 rows
[12/229] HmbTG_ALKIS_260115_044von283.xml ... 10 rows
[13/229] HmbTG_ALKIS_260115_045von283.xml ... 2 rows
[14/229] HmbTG_ALKIS_260115_046von283.xml ... 2 rows
[15/229] HmbTG_ALKIS_260115_047von283.xml ... 764 rows
[16/229] HmbTG_ALKIS_260115_048von283.xml ... 3260 rows
[17/229] HmbTG_ALKIS_260115_049von283.xml ... 4937 rows
[18/229] HmbTG_ALKIS_260115_050von283.xml ... 339 rows
[19/229] HmbTG_ALKIS_260

In [7]:
def fix_mixed_column(series):
    """Flatten arrays/ndarrays, then cast to the most appropriate scalar type."""
    def scalar(val):
        if isinstance(val, (list, np.ndarray)):
            arr = list(val)
            return arr[0] if len(arr) > 0 else None
        return val
    
    s = series.apply(scalar)
    # try numeric first, fall back to string
    numeric = pd.to_numeric(s, errors="coerce")
    if numeric.notna().sum() > 0 and numeric.isna().sum() == s.isna().sum():
        return numeric  # clean numeric column
    return s.astype(str).where(s.notna(), None)  # force to string, keep NaN as None

def clean_up_and_fix_columns(gdf: gpd.GeoDataFrame, ) -> gpd.GeoDataFrame:
    # test each object column against pyarrow before saving
    for col in gdf.select_dtypes(include="object").columns:
        if col == "geometry":
            continue
        try:
            pa.array(gdf[col].tolist(), from_pandas=True)
        except pa.ArrowInvalid or pa.ArrowTypeError:
            print(f"Fixing: {col}")
            gdf[col] = fix_mixed_column(gdf[col])

    return gdf


In [8]:
gdf = clean_up_and_fix_columns(gdf)
gdf.to_parquet("buildings_cleaned_up.parquet")

ArrowTypeError: Expected bytes, got a 'numpy.ndarray' object

In [ ]:
gdf = gpd.read_parquet("buildings_cleaned_up.parquet")
print(gdf.shape)
print(gdf.crs)
print(gdf.columns.tolist())
print(gdf.head())